In [1]:
import cv2
import os

# cropping


In [4]:


# =====================================================
# Select Dataset
# =====================================================

choice = input("Enter dataset (train / augmented / valid): ").strip().lower()

if choice == "train":
    image_folder = r"..\data_Set\train\images"
    label_folder = r"..\data_Set\train\labels"

elif choice == "augmented":
    image_folder = r"..\data_Set\train\augmented_images\images"
    label_folder = r"..\data_Set\train\augmented_images\labels"

elif choice in ["valid", "val", "validation"]:
    image_folder = r"..\data_Set\validation\images"
    label_folder = r"..\data_Set\validation\labels"

else:
    raise ValueError("Invalid choice! Please enter train, augmented or valid.")

# =====================================================
# Output Folder
# =====================================================

output_folder = os.path.join(
    r"..\data_set_generator_for_cnns_only",
    choice
)

os.makedirs(output_folder, exist_ok=True)

# =====================================================
# Supported Extensions
# =====================================================

extensions = (".jpg", ".jpeg", ".png", ".bmp")

# =====================================================
# Crop Images
# =====================================================

saved = 0
missing_labels = 0

for image_name in os.listdir(image_folder):

    if not image_name.lower().endswith(extensions):
        continue

    image_path = os.path.join(image_folder, image_name)

    image = cv2.imread(image_path)

    if image is None:
        print(f"Could not read {image_name}")
        continue

    H, W = image.shape[:2]

    label_name = os.path.splitext(image_name)[0] + ".txt"
    label_path = os.path.join(label_folder, label_name)

    if not os.path.exists(label_path):
        print(f"Label not found: {label_name}")
        missing_labels += 1
        continue

    with open(label_path, "r") as f:
        lines = f.readlines()

    for idx, line in enumerate(lines):

        values = line.strip().split()

        if len(values) != 5:
            continue

        class_id, xc, yc, bw, bh = map(float, values)

        # YOLO -> Pixel Coordinates
        xc *= W
        yc *= H
        bw *= W
        bh *= H

        x1 = int(xc - bw / 2)
        y1 = int(yc - bh / 2)
        x2 = int(xc + bw / 2)
        y2 = int(yc + bh / 2)

        # Keep coordinates inside image
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(W, x2)
        y2 = min(H, y2)

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        save_name = f"{os.path.splitext(image_name)[0]}_{idx}.png"
        save_path = os.path.join(output_folder, save_name)

        cv2.imwrite(save_path, crop)

        saved += 1

print("\n==============================")
print(f"Dataset       : {choice}")
print(f"Images Saved  : {saved}")
print(f"Missing Label : {missing_labels}")
print(f"Output Folder : {output_folder}")
print("==============================")


Dataset       : augmented
Images Saved  : 279
Missing Label : 0
Output Folder : ..\data_set_generator_for_cnns_only\augmented
